# CDK20への高活性化合物のフレキシブルドッキング(gnina)

## Why

`cdk20_similar_targets.ipynb`でCDK20の代替として選んだ6つのデータリッチな
パラログ(CDK2/9/1/6/5/7)について、`cdk_paralogs_active_compounds.ipynb`で
`pchembl_mean >= 8.8`の高活性化合物113個を集めた。これらは他のCDKに対する
活性実績があるので、CDK20自身にも通用するかを見るため、`cdk20_pocket.ipynb`
で見つけたCDK20 AlphaFold構造の推定ポケット(druggability_score=0.646)に
実際にドッキングしてみる。

CDK20には実験構造もリガンドも一切ないので、ポケットの側鎖がリガンドに
応じてどう動くかは分からない -- そこで側鎖の一部をフレキシブルにした
ドッキング(gnina 1.3.2、GPU CNN rescoring)を行う。

## 1. Receptor: rigid + flexible side chains (meeko)

`cdk20_pocket.ipynb`のポケット(24残基、`list_pockets`の上位ポケット)を
そのまま再現するところから始める(自己完結のため、他ノートブックの実行結果
には依存しない)。

**フレキシブル残基は24個全部ではなく6個に絞る**: 一度24残基すべてを
`meeko`の`--flexres`でフレキシブルにしてドッキングを試したところ、
exhaustiveness=1でも2分以上経っても終わらなかった(24残基分の回転可能結合
+ このポケットサイズの探索空間が大きすぎるため)。ポケット中心への
CA距離が近い上位6残基(側鎖が動く残基のみ、GLY/ALA/PROなど動かせない
残基は`mk_prepare_receptor.py`が自動的に除外)に絞ったところ、1化合物
あたり約48秒に収まった。

In [ ]:
import os

import numpy as np
import pandas as pd

from chem import protein

AF_STRUCTURE = "cdk20_af_data/AF-Q8IZL9-F1.pdb"
DOCKING_OUTDIR = "cdk20_docking_data"
os.makedirs(DOCKING_OUTDIR, exist_ok=True)

pockets = protein.list_pockets(AF_STRUCTURE)
top_pocket = pockets[0]
print(f"pocket {top_pocket['pocket_id']} (druggability_score={top_pocket['druggability_score']}), "
      f"{len(top_pocket['residues'])} lining residues")


def docking_box(pocket, padding=4.0):
    """Axis-aligned (center, size) docking box around a list_pockets/find_pocket
    result's alpha spheres, in Angstroms -- ready for AutoDock Vina/gnina's
    center_x/y/z + size_x/y/z. Same helper as cdk20_pocket.ipynb.
    """
    spheres = pocket["spheres"]
    centers = np.array([[s["x"], s["y"], s["z"]] for s in spheres])
    radii = np.array([s["radius"] for s in spheres])
    mins = (centers - radii[:, None]).min(axis=0)
    maxs = (centers + radii[:, None]).max(axis=0)
    center = (mins + maxs) / 2
    size = (maxs - mins) + 2 * padding
    return {
        "center_x": round(float(center[0]), 2), "center_y": round(float(center[1]), 2),
        "center_z": round(float(center[2]), 2),
        "size_x": round(float(size[0]), 2), "size_y": round(float(size[1]), 2),
        "size_z": round(float(size[2]), 2),
    }


BOX = docking_box(top_pocket)
print("docking box:", BOX)

In [ ]:
from Bio.PDB import PDBParser

parser = PDBParser(QUIET=True)
structure = parser.get_structure("cdk20", AF_STRUCTURE)
chain = next(structure.get_models())["A"]
box_center = np.array([BOX["center_x"], BOX["center_y"], BOX["center_z"]])

# GLY/ALA/PRO have no rotatable side-chain bond (GLY: no CB; ALA: CB has nothing
# further to rotate; PRO: ring is fixed) -- mk_prepare_receptor.py drops these
# silently ("has no movable atoms"), so exclude them here too, before ranking by
# distance, or the closest-N selection ends up with fewer real flexible residues
# than intended (this happened: an earlier version of this cell picked 3 GLY/ALA
# residues among its top 6, leaving only 3 actually-flexible side chains).
NO_MOVABLE_SIDECHAIN = {"GLY", "ALA", "PRO"}

pocket_resnums = sorted({r["resnum"] for r in top_pocket["residues"]})
dists = []
for resnum in pocket_resnums:
    res = chain[resnum]
    if "CA" not in res or res.get_resname() in NO_MOVABLE_SIDECHAIN:
        continue
    d = np.linalg.norm(res["CA"].coord - box_center)
    dists.append((resnum, res.get_resname(), round(float(d), 2)))
dists.sort(key=lambda x: x[2])

N_FLEX = 6
flex_residues = [resnum for resnum, _, _ in dists[:N_FLEX]]
print(f"{N_FLEX} flexible residues (closest to pocket center, GLY/ALA/PRO excluded):")
for resnum, resname, d in dists[:N_FLEX]:
    print(f"  A:{resnum} {resname}  ({d} A from pocket center)")

In [ ]:
import subprocess

flexres_arg = ",".join(f"A:{r}" for r in flex_residues)
receptor_basename = os.path.join(DOCKING_OUTDIR, "cdk20")

subprocess.run(
    [
        "mk_prepare_receptor.py",
        "--read_pdb", AF_STRUCTURE,
        "-f", flexres_arg,
        "-o", receptor_basename,
        "-p",
    ],
    check=True,
)

RIGID_PDBQT = receptor_basename + "_rigid.pdbqt"
FLEX_PDBQT = receptor_basename + "_flex.pdbqt"
print("wrote", RIGID_PDBQT, "and", FLEX_PDBQT)

## 2. Ligands: the 113 potent compounds, as PDBQT (RDKit + meeko)

`cdk_paralogs_active_compounds.ipynb`で書き出した各ターゲットのChEMBL
tsvから、同じ条件(`pchembl_mean >= 8.8`)で113化合物を再現する(こちらも
自己完結、tsvファイルはディスク上に残っているのでノートブックの実行状態
には依存しない)。各化合物をRDKitで3D化(embed + MMFF最適化)し、meekoで
PDBQTに変換する。

In [ ]:
TARGETS = ["CDK2_HUMAN", "CDK9_HUMAN", "CDK1_HUMAN", "CDK6_HUMAN", "CDK5_HUMAN", "CDK7_HUMAN"]
PCHEMBL_THRESHOLD = 8.8
CHEMBL_OUTDIR = "cdk_paralogs_chembl_data"

top_compounds = []
for target in TARGETS:
    df = pd.read_csv(os.path.join(CHEMBL_OUTDIR, f"{target}.tsv"), sep="\t")
    top = df[df["pchembl_mean"] >= PCHEMBL_THRESHOLD].sort_values("pchembl_mean", ascending=False).copy()
    top.insert(0, "target", target)
    top_compounds.append(top)

candidates_df = pd.concat(top_compounds, ignore_index=True)
print(f"{len(candidates_df)} candidate compounds across {len(TARGETS)} targets")
candidates_df.to_csv(os.path.join(DOCKING_OUTDIR, "candidate_compounds.tsv"), sep="\t", index=False)

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from meeko import MoleculePreparation, PDBQTWriterLegacy

LIGANDS_DIR = os.path.join(DOCKING_OUTDIR, "ligands")
os.makedirs(LIGANDS_DIR, exist_ok=True)

# rigid_macrocycles=True: meeko's default macrocycle handling opens large rings
# (min_ring_size=7) into flexible pseudo-atoms (atom types like "CG0"), which
# gnina's Vina-derived atom typer rejects outright ("... is not a valid AutoDock
# type"). A handful of these 113 compounds have a macrocyclic ring and hit this
# -- keeping macrocycles rigid avoids it (loses some macrocycle flexibility, but
# every other rotatable bond, including all six flexible side chains, is
# unaffected).
n_ok, n_fail = 0, 0
for i, row in enumerate(candidates_df.itertuples(), 1):
    out_path = os.path.join(LIGANDS_DIR, f"{row.target}__{row.parent_chembl_id}.pdbqt")
    if os.path.exists(out_path):
        n_ok += 1
        continue
    mol = Chem.MolFromSmiles(row.smiles)
    if mol is None or AllChem.EmbedMolecule(Chem.AddHs(mol), randomSeed=1) < 0:
        print(f"[{i}/{len(candidates_df)}] {row.target} {row.parent_chembl_id}: failed to prepare 3D structure")
        n_fail += 1
        continue
    mol = Chem.AddHs(mol)
    AllChem.EmbedMolecule(mol, randomSeed=1)
    AllChem.MMFFOptimizeMolecule(mol)
    setups = MoleculePreparation(rigid_macrocycles=True).prepare(mol)
    pdbqt_string, ok, err = PDBQTWriterLegacy.write_string(setups[0])
    if not ok:
        print(f"[{i}/{len(candidates_df)}] {row.target} {row.parent_chembl_id}: meeko failed: {err}")
        n_fail += 1
        continue
    with open(out_path, "w") as f:
        f.write(pdbqt_string)
    n_ok += 1
    if i % 20 == 0 or i == len(candidates_df):
        print(f"[{i}/{len(candidates_df)}] prepared")

print(f"done: {n_ok} ok, {n_fail} failed")

## 3. Batch flexible docking with gnina (GPU CNN rescoring)

`gnina`はビルド済みバイナリ(`~/tools/gnina/gnina.1.3.2`)を直接呼び出す
(pip/condaパッケージではないため)。`libcudnn.so.9`が必要なので、専用の
conda環境`gnina`(`cudnn=9.10.2.21`)のlibディレクトリを`LD_LIBRARY_PATH`
で指定する。

- デフォルトのCNNアンサンブルモデル(`--cnn`省略時)はこのビルドでは
  読み込みエラーになる(`Could not read torch model dense_1_3`)ため、
  単一モデル`crossdock_default2018`を明示指定する。
- `exhaustiveness=8`(gninaのデフォルト)は`exhaustiveness=1`とほぼ同じ
  実行時間(16 CPUで並列化されるため)なので、品質のためデフォルト値を使う。
- 既にドッキング済み(出力sdfが存在する)化合物はスキップする -- 113化合物
  で約90分かかるため、中断・再開できるようにするため。

In [ ]:
import glob
import time

GNINA = os.path.expanduser("~/tools/gnina/gnina.1.3.2")
GNINA_ENV = dict(os.environ, LD_LIBRARY_PATH="/opt/miniforge3/envs/gnina/lib:" + os.environ.get("LD_LIBRARY_PATH", ""))
POSES_DIR = os.path.join(DOCKING_OUTDIR, "poses")
os.makedirs(POSES_DIR, exist_ok=True)

ligand_files = sorted(glob.glob(os.path.join(LIGANDS_DIR, "*.pdbqt")))
print(f"{len(ligand_files)} ligands to dock")

t0 = time.time()
for i, lig_path in enumerate(ligand_files, 1):
    name = os.path.splitext(os.path.basename(lig_path))[0]
    out_sdf = os.path.join(POSES_DIR, f"{name}.sdf")
    out_flex = os.path.join(POSES_DIR, f"{name}_flex.pdbqt")

    if os.path.exists(out_sdf):
        continue

    cmd = [
        GNINA,
        "-r", RIGID_PDBQT, "--flex", FLEX_PDBQT,
        "-l", lig_path,
        "--center_x", str(BOX["center_x"]), "--center_y", str(BOX["center_y"]), "--center_z", str(BOX["center_z"]),
        "--size_x", str(BOX["size_x"]), "--size_y", str(BOX["size_y"]), "--size_z", str(BOX["size_z"]),
        "--cnn", "crossdock_default2018", "--cnn_scoring", "rescore",
        "--exhaustiveness", "8", "--num_modes", "5",
        "-o", out_sdf, "--out_flex", out_flex,
        "--verbosity", "1",
    ]
    elapsed = time.time() - t0
    print(f"[{i}/{len(ligand_files)}] {name}: docking... ({elapsed:.0f}s elapsed so far)")
    result = subprocess.run(cmd, env=GNINA_ENV, capture_output=True, text=True, timeout=300)
    if result.returncode != 0:
        print(f"[{i}/{len(ligand_files)}] {name}: FAILED (exit {result.returncode})")
        print(result.stderr[-2000:])

print(f"done. total time this run: {time.time() - t0:.0f}s")

## 4. Aggregating results

In [ ]:
rows = []
n_unparsed = 0
for sdf_path in sorted(glob.glob(os.path.join(POSES_DIR, "*.sdf"))):
    name = os.path.splitext(os.path.basename(sdf_path))[0]
    target, chembl_id = name.split("__")
    try:
        mol = next(iter(Chem.SDMolSupplier(sdf_path)), None)
    except OSError:
        mol = None
    if mol is None:
        n_unparsed += 1
        continue
    rows.append(
        {
            "target": target,
            "parent_chembl_id": chembl_id,
            "CNNscore": float(mol.GetProp("CNNscore")),
            "CNNaffinity": float(mol.GetProp("CNNaffinity")),
            "minimizedAffinity": float(mol.GetProp("minimizedAffinity")),
        }
    )

results_df = pd.DataFrame(rows)
results_df = results_df.merge(
    candidates_df[["target", "parent_chembl_id", "smiles", "pchembl_mean"]],
    on=["target", "parent_chembl_id"],
    how="left",
)
results_df = results_df.sort_values("CNNscore", ascending=False).reset_index(drop=True)
# n_unparsed is 1 compound (CDK6_HUMAN/CHEMBL5755607, a guanidine-containing
# molecule) whose docked pose fails RDKit sanitization ("Explicit valence for atom
# N ... greater than permitted") -- a bond-order perception quirk isolated to this
# one compound's guanidinium group, not a pipeline-wide issue.
print(f"{len(results_df)} docked and parsed successfully ({n_unparsed} excluded due to a pose sanitization failure)")
display(
    results_df.head(20)[["target", "parent_chembl_id", "pchembl_mean", "CNNscore", "CNNaffinity", "minimizedAffinity"]]
    .style.hide(axis="index")
    .format({"pchembl_mean": "{:.2f}", "CNNscore": "{:.3f}", "CNNaffinity": "{:.2f}", "minimizedAffinity": "{:.2f}"})
)

### Does docking score agree with known potency?

`pchembl_mean`は他のCDKパラログに対する実験的な活性データ(CDK20自身の
データではない)。もしCNNscoreと`pchembl_mean`に相関があれば、この
ドッキング設定(ポケット・フレキシブル残基・ボックス)がある程度妥当な
シグナルを捉えていると考えられる(相関がなくても、CDK20自身の活性が
そもそも異なる可能性があるので、これだけで結論は出せない)。

In [ ]:
corr = results_df[["pchembl_mean", "CNNscore"]].corr().iloc[0, 1]
print(f"Pearson correlation (pchembl_mean vs CNNscore): {corr:.3f}")

import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
for target, group in results_df.groupby("target"):
    ax.scatter(group["pchembl_mean"], group["CNNscore"], label=target, alpha=0.7)
ax.set_xlabel("pchembl_mean (activity vs. other CDKs)")
ax.set_ylabel("CNNscore (gnina, CDK20 pocket)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Visualizing the top pose

In [ ]:
best = results_df.iloc[0]
best_flex_pdbqt = os.path.join(POSES_DIR, f"{best['target']}__{best['parent_chembl_id']}_flex.pdbqt")
best_sdf = os.path.join(POSES_DIR, f"{best['target']}__{best['parent_chembl_id']}.sdf")
print(
    f"Best pose: {best['target']} {best['parent_chembl_id']} "
    f"(CNNscore={best['CNNscore']:.3f}, pchembl_mean={best['pchembl_mean']:.2f})"
)

import py3Dmol
from IPython.display import display

with open(AF_STRUCTURE) as f:
    rec_pdb = f.read()
with open(best_sdf) as f:
    lig_sdf = f.read()
with open(best_flex_pdbqt) as f:
    flex_pdbqt = f.read()

view = py3Dmol.view(width=650, height=500)
view.addModel(rec_pdb, "pdb")
view.setStyle({"cartoon": {"color": "lightgrey", "opacity": 0.6}})
view.addStyle({"resi": flex_residues}, {"stick": {"colorscheme": "greenCarbon"}})
view.addModel(flex_pdbqt, "pdbqt")
view.setStyle({"model": 1}, {"stick": {"colorscheme": "yellowCarbon"}})
view.addModel(lig_sdf, "sdf")
view.setStyle({"model": 2}, {"stick": {"colorscheme": "magentaCarbon"}})
view.zoomTo({"resi": flex_residues})
view.show()

## Summary

113個の高ポテンシー化合物(他のCDKパラログに対して`pchembl_mean >= 8.8`)
を、CDK20 AlphaFoldモデルの推定ポケット(側鎖6残基フレキシブル: ASP145,
ASN132, GLU12, VAL18, LYS33, HIS15)にgninaでドッキングした。112/113が
正常に解析できた(1化合物はポーズのRDKit sanitizationに失敗し除外)。

`pchembl_mean`(他のCDKパラログに対する実験活性)とCNNscoreのPearson相関は
0.276 -- 弱いが正の相関があり、このドッキング設定が全くの的外れではない
ことを示唆する(強い相関ではないので、これだけでCDK20自身への活性を
断定はできない)。

最上位ヒットは`CDK9_HUMAN`由来の`CHEMBL1986943`(CNNscore=0.985、
pchembl_mean=8.90)。上位ヒットは今後の実験検証・追加解析の候補となる。